# M6 Original、Local-scale 與 rolling IFOMC：四種資料比較

這份 notebook 只擬合一次 M6 點預測，再對同一組點預測比較 Original residual-window、Local-scale residual-window 與 rolling IFOMC residual interval。IFOMC 在發出當期區間後，才用已觀察到的當期真值更新轉移矩陣。M1M9–GARCH 使用 seeds 2020–2039 的 20 次配對 Monte Carlo；其餘三組為單次實證比較。

In [ ]:
from pathlib import Path
import sys, time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src'/'kf_forecasting').exists()), None)
if ROOT is None: raise RuntimeError('找不到 project/meeting 根目錄')
sys.path.insert(0, str(ROOT/'src'))

from kf_forecasting.models.kf_enbpi import EnbPIConfig, simulate_m1_m9_garch_low_constant_high_variance_data
from kf_forecasting.models.kf_out_of_time_enbpi import OutOfTimeEnbPIConfig, locally_scaled_from_fitted, run_out_of_time_enbpi
from kf_forecasting.models.kf_ifomc_enbpi import IFOMCIntervalConfig, rolling_ifomc_from_fitted

ALPHA=0.05; SEED=2026
pd.set_option('display.max_columns', None)
print('ROOT =', ROOT)

## 共用執行與評估函數

In [ ]:
def metrics(frame, alpha=ALPHA):
    y,p,l,u=(frame[c].to_numpy(float) for c in ['truth','point','lower','upper']); w=u-l
    lm=y<l; um=y>u; score=w+2/alpha*((l-y)*lm+(y-u)*um)
    return {'RMSE':np.sqrt(np.mean((y-p)**2)), 'Coverage (%)':100*np.mean((y>=l)&(y<=u)),
            'Mean width':np.mean(w), 'Median width':np.median(w),
            'Lower miss (%)':100*np.mean(lm), 'Upper miss (%)':100*np.mean(um),
            'Interval score':np.mean(score)}

def transformed(frame, inverse):
    out=frame[['date','method','truth','point','lower','upper']].copy()
    for c in ['truth','point','lower','upper']: out[c]=inverse(out[c].to_numpy(float))
    return out

def run_three_intervals(y, train_size, dates, *, seed=SEED, model_name='M6', inverse=lambda x:x, base_config=None):
    base = base_config or EnbPIConfig(alpha=ALPHA, random_state=seed)
    cfg=OutOfTimeEnbPIConfig(base=base)
    t0=time.perf_counter(); fitted=run_out_of_time_enbpi(np.asarray(y,float),train_size,config=cfg,model_name=model_name,data_seed=seed); fit_sec=time.perf_counter()-t0
    t0=time.perf_counter(); local=locally_scaled_from_fitted(fitted,pd.Index(dates),neighbours=60,support_lower=None); local_sec=time.perf_counter()-t0
    t0=time.perf_counter(); ifomc=rolling_ifomc_from_fitted(fitted,pd.Index(dates),config=IFOMCIntervalConfig(n_states=10,min_source_transitions=10)); ifomc_sec=time.perf_counter()-t0
    f=fitted.forecast
    original=pd.DataFrame({'date':dates,'method':'Original','truth':f.truth,'point':f.point,'lower':f.lower,'upper':f.upper})
    local_frame=local.predictions.copy(); local_frame['method']='Local-scale'
    ifomc_frame=ifomc.predictions.copy(); ifomc_frame['method']='Rolling IFOMC'
    frames=[transformed(original,inverse),transformed(local_frame,inverse),transformed(ifomc_frame,inverse)]
    summary=pd.DataFrame([{'version':d.method.iloc[0],**metrics(d)} for d in frames])
    summary['M6 fit sec']=fit_sec; summary['postprocess sec']=[0.,local_sec,ifomc_sec]; summary['total sec']=summary['M6 fit sec']+summary['postprocess sec']
    return summary,pd.concat(frames,ignore_index=True),fitted,local,ifomc

def plot_three(predictions,title):
    fig,axes=plt.subplots(3,1,figsize=(15,10),sharex=True)
    for ax,(name,d) in zip(axes,predictions.groupby('method',sort=False)):
        ax.plot(d.date,d.truth,'k',lw=1,label='Truth'); ax.plot(d.date,d.point,color='tab:blue',lw=1,label='Point')
        ax.fill_between(d.date,d.lower,d.upper,color='tab:blue',alpha=.18,label='95% interval'); ax.set_title(name); ax.grid(alpha=.25)
    axes[0].legend(ncol=3); fig.suptitle(title); fig.tight_layout(); plt.show()

## 實驗一：M1M9–GARCH（20 次 Monte Carlo）

In [ ]:
TRAIN=650; H=200; MC_SEEDS=range(2020,2040); mc_summary=[]; representative=None
for seed in MC_SEEDS:
    _,_,y=simulate_m1_m9_garch_low_constant_high_variance_data(TRAIN+H,rng=np.random.default_rng(seed))
    s,p,*_=run_three_intervals(y,TRAIN,pd.Index(np.arange(TRAIN,TRAIN+H)),seed=seed,model_name=f'M1M9 seed {seed}')
    s.insert(0,'seed',seed); mc_summary.append(s)
    if seed==2026: representative=p.copy()
    print(f'finished seed {seed}')
m1m9_by_seed=pd.concat(mc_summary,ignore_index=True)
metric_cols=['RMSE','Coverage (%)','Mean width','Median width','Lower miss (%)','Upper miss (%)','Interval score','total sec']
m1m9_summary=m1m9_by_seed.groupby('version')[metric_cols].agg(['mean','std'])
display(m1m9_summary.round(3)); plot_three(representative,'M1M9–GARCH：seed 2026')

## 實驗二：0050

In [ ]:
close=pd.read_csv(ROOT/'data/raw/0050_adjusted_close.csv',parse_dates=['Date']).set_index('Date').adjusted_close
train=close.loc['2020':'2024']; test=close.loc['2025-01-01':'2026-07-31']; y=np.log(pd.concat([train,test]).to_numpy(float))
tw0050_summary,tw0050_predictions,*_=run_three_intervals(y,len(train),test.index,model_name='0050',inverse=np.exp)
display(tw0050_summary.round(3)); plot_three(tw0050_predictions,'0050')

## 實驗三：太陽黑子

In [ ]:
cols=['year','month','decimal_date','sunspot','std','observations','provisional']
d=pd.read_csv(ROOT/'data/raw/SN_m_tot_V2.0.csv',sep=';',header=None,names=cols); d['date']=pd.to_datetime(dict(year=d.year,month=d.month,day=1))
series=d.loc[d.date.between('1900-01-01','2023-01-01')&d.sunspot.ge(0)].set_index('date').sunspot.astype(float).sort_index(); n=int(.8*len(series)); test=series.iloc[n:]
sunspot_summary,sunspot_predictions,*_=run_three_intervals(series.to_numpy(),n,test.index,model_name='Sunspot')
display(sunspot_summary.round(3)); plot_three(sunspot_predictions,'Monthly sunspot')

## 實驗四：風速 Data1

In [ ]:
wind_path=ROOT/'data/raw/wind_speed_data1.csv'
if not wind_path.exists():
    url='https://raw.githubusercontent.com/hyuzhou/Wind-speed-forecasting-/master/Data/Data1.csv'
    wind_path.parent.mkdir(parents=True,exist_ok=True); pd.read_csv(url).to_csv(wind_path,index=False)
wind_data=pd.read_csv(wind_path); wind_col=next(c for c in wind_data.columns if c.strip().lower()=='wind speed')
wind=wind_data[wind_col].astype(float).dropna().reset_index(drop=True); test_size=100; train_size=len(wind)-test_size
wind_base=EnbPIConfig(window_size=15,alpha=ALPHA,n_bootstrap=30,batch_size=1,beta_grid_size=101,oob_bias_correction=True,oob_bias_correction_mode='combined',arima_order=None,arima_max_p=4,arima_max_q=4,arima_max_iter=500,ann_hidden_layers=(32,16),ann_max_iter=500,ann_alpha=1e-4,ann_learning_rate_init=1e-3,ann_target_standardization=True,ann_early_stopping=False,ann_rolling_validation=True,ann_rolling_splits=3,ann_validation_fraction=.10,ann_tol=1e-3,random_state=SEED,process_variance=.5,measurement_variance=10.)
wind_summary,wind_predictions,wind_fit,wind_local,wind_ifomc=run_three_intervals(wind.to_numpy(),train_size,pd.RangeIndex(1,test_size+1,name='test_step'),model_name='Wind Data1',base_config=wind_base)
display(wind_summary.round(3)); display(wind_ifomc.state_table.round(3)); display(wind_ifomc.final_transition_matrix.round(3)); plot_three(wind_predictions,'Wind-speed Data1')

## 輸出說明

每組 summary 包含 RMSE、coverage、mean/median width、上下界失誤率、interval score 與運行時間。圖形各有 Original、Local-scale 與 rolling IFOMC 三列。風速區塊另顯示 IFOMC 的狀態邊界與 testing 更新後的最終轉移矩陣。Notebook 不會自動寫入 results 資料夾，執行結果會保留在 ipynb 內。